In [4]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\karth\AppData\Local\Temp\ipykernel_9740\1975300343.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
c:\Users\karth\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

NameError: name 'Path' is not defined

In [4]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [6]:

chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 8 chunks

Example chunk:
Content: SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE
 LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE
 INSTRUMENTS ACT, 1881
Notice No.
LL-TEST-2026-001
Date
14 August 2026
To
Mr. Rohan Mehta
Address
24,...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

Embeddings and vectorDB

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\karth\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3696.12it/s]


Model loaded successfully. Embedding dimension: 384
